# Практическое занятие 2.2

Реализовать алгоритм Кросс-Энтропии для непрерывного пространства действий. Обучить агента решать Pendulum-v1 или MountainCarContinuous-v0 на выбор. Исследовать гиперпараметры алгоритма и выбрать лучшие.

# Установка зависимостей

In [ ]:
%%capture

!pip install wandb

# Для рендеринга
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install -U colabgymrender
!pip install imageio==2.4.1
!pip install --upgrade AutoROM
!AutoROM --accept-license
!pip install gym[atari,accept-rom-license] == 0.25.1
!pip install opencv-python
!pip install tqdm

# Библиотеки

In [1]:
from typing import Optional, Any, Literal
from warnings import filterwarnings
import random
import time
import os

import torch.nn as nn
import torch.nn.functional as F
import torch

import scipy.stats as stats
import numpy as np
import wandb

from tqdm import tqdm
import gym
from colabgymrender.recorder import Recorder

/home/merci/PycharmProjects/pytorchprojects/deep-RL-course/env/lib/python3.10/site-packages/colabgymrender/recorder.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import Video, display


# Агент

In [2]:
class CrossEntropyAgent(nn.Module):
    def __init__(self,
                 state_dim: int,
                 action_n: int,
                 depth: list[int, ...],
                 dropout: float = None,
                 batch_norm: bool = False,
                 device: Literal["cpu", "cuda"] = "cuda",
                 epsilan: float = 0,
                 gamma: float = 0
                 ):
        super().__init__()
        self.__iterations = 0
        self.gamma = gamma
        self.state_dim = state_dim
        self.dropout = dropout
        self.batch_norm = batch_norm
        self.action_n = action_n
        self.depth = depth
        self.layers = [state_dim, *depth, action_n]
        self.head_block_index = len(self.layers) - 2
        self.device = torch.device(device)
        self.stats = {
            "noise": []
        }

        self.__uniform_policy = np.full(self.action_n, (1.0 / self.action_n))
        self.deep_q_network = nn.Sequential(*[
            self.__get_block(index, in_features, out_features)
            for index, (in_features, out_features) in enumerate(zip(self.layers, self.layers[1:]))
        ])

        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.0035)
        self.criterion = nn.MSELoss()

        self.to_ndarray = lambda x: x.detach().cpu().numpy()
        self.to_cuda_tenzor = lambda x: torch.FloatTensor(x).to(self.device)
        self.to_list = lambda x:  list(self.to_ndarray(x.flatten()))
        self.epsilan = epsilan # std

    def exponential_epsilan_sheduler_step(self) -> None:
      self.__iterations += 1
      self.epsilan /= np.exp(self.__iterations * self.gamma)

    def __get_block(self,
                    index: int,
                    in_features: int,
                    out_features: int
                    ) -> nn.Sequential:

        _block = nn.Sequential(nn.Linear(in_features, out_features))
        if index != self.head_block_index:
            _block.append(nn.ReLU())
            if self.batch_norm:
                _block.append(nn.BatchNorm1d(out_features))
            if self.dropout:
                _block.append(nn.Dropout(self.dropout))
        return _block.to(self.device)

    def forward(self, x: torch.Tensor | np.ndarray | list, noising: bool = False) -> torch.FloatTensor:
      if not isinstance(x, torch.Tensor):
        x = self.to_cuda_tenzor(x)
      x = self.deep_q_network(x)

      if noising:
        noise = torch.normal(mean=0, std=self.epsilan, size=(1,))
        self.stats["noise"].append(noise)
        x += self.to_cuda_tenzor(noise)
      return F.tanh(x)

    def get_action(self, state: torch.Tensor | np.ndarray | list) -> float:
      """
      Возвращает действие агента A при условии состояния S
      :param state: текущее состояние
      :return: P(A | S)
      """

      with torch.no_grad():
        output = self.forward(state, True)
      output = self.to_ndarray(output)
      return output

    def step(self, elite_trajectories: list) -> float:
      self.deep_q_network.train()
      elite_states = []
      elite_actions = []

      for trajectory in elite_trajectories:
        elite_states.extend(trajectory["states"])
        elite_actions.extend(trajectory["actions"])

      elite_states = self.to_cuda_tenzor(elite_states)
      elite_actions = self.to_cuda_tenzor(elite_actions)

      preds = self.forward(elite_states, False)
      loss = self.criterion(preds, elite_actions)

      loss.backward()
      self.optimizer.step()
      self.exponential_epsilan_sheduler_step()
      self.optimizer.zero_grad()
      self.deep_q_network.eval()
      return loss.item()

# Среда

In [3]:
class SandBoxEnvironment:
  def __init__(self,
                stage: Literal["train", "test"],
                agent: CrossEntropyAgent,
                env_name: str,
                render_mode: Optional[str] = "human",
                **env_kwargs
                ):
    env = gym.make(env_name, **env_kwargs)
    self.env = env if stage == "train" else Recorder(env, f"./{stage}-videos")
    self.stage = stage
    self.agent = agent
    self.render_mode = render_mode

    self.current_iteration = 0
    self.save_top_k = None
    self.save_by = None
    self.iterations = None
    self.start_time = None
    self.mode = None
    self.discont_gamma = None

    self.models = []
    self.rewards = self.init_rewards()

    self.total_rewards = lambda trajectories: [trajectory["total_rewards"] for trajectory in trajectories]
    self.iter_mean_rewards = lambda trajectories: np.round(np.mean(self.total_rewards(trajectories)), 2)
    self.time = lambda: round(time.time() - self.start_time, 2) if self.start_time else -1

  def init_rewards(self) -> dict:
    return {
        "elite": [],
        "all": [],
        "loss": [],
        "epsilan": [],
        "q_param": [],
    }

  def define_checkpoint(self) -> dict:
    best_finder = np.argmax if self.mode == "max" else np.argmin
    best_state_index: int = best_finder([
        model[self.save_by]
        for model in self.models
        ])

    metrics = [
        f"{key}={value}"
        for key, value in self.models[best_state_index].items()
        if key != "state_dict"
        ]

    filename = "_".join(metrics) + ".pth"
    state_dict = self.models[best_state_index]["state_dict"]
    path = os.path.join(os.getcwd(), "models", filename)
    return {
        "filepath": path,
        "state_dict": state_dict
    }


  def model_checkpoint(self) -> None:
    save_each = self.iterations // self.save_top_k
    if self.current_iteration % save_each == 0:
      checkpoint_dict = self.define_checkpoint()
      torch.save(checkpoint_dict["state_dict"], checkpoint_dict["filepath"])
      self.models.clear()

  def __get_trajectory(self, trajectory_len: int = 500) -> dict:
    state = self.env.reset()  # Инициализация среды и получения начального состояния агента
    trajectory = {
        "states": [],
        "actions": [],
        "total_rewards": 0
    }

    for t in range(trajectory_len):
        trajectory["states"].append(state)
        action = self.agent.get_action(state)  # Агент совершает действие P(A | S)
        trajectory["actions"].append(action)
        state, reward, terminated, truncated = self.env.step(action)  # Среда обновляется
        trajectory["total_rewards"] += (reward * (self.discont_gamma ** t))
        if terminated:
          break
    return trajectory

  def get_trajectories(self, trajectory_len: int, trajectory_n: int) -> list:
    return [
        self.__get_trajectory(trajectory_len)
        for _ in range(trajectory_n)
    ]

  def get_elite_trajectories(self, trajectories: list, q_param: Optional[float] = 0.85) -> list:
    total_rewards = self.total_rewards(trajectories)
    quantile = np.quantile(total_rewards, q_param)
    return [
        trajectory
        for trajectory in trajectories
        if trajectory["total_rewards"] >= quantile
    ]

  def on_epoch_end(self, trajectories: list, elite_trajectories: list) -> None:
    min_elite = np.min(self.total_rewards(elite_trajectories))
    max_elite = np.max(self.total_rewards(elite_trajectories))
    mean_reward = self.iter_mean_rewards(trajectories)
    elite_mean_reward = self.iter_mean_rewards(elite_trajectories)
    time = self.time()

    self.rewards["all"].append(mean_reward)
    self.rewards["elite"].append(elite_mean_reward)

    loss = self.rewards["loss"][~0]
    q_param = self.rewards["q_param"][~0]
    epsilan = self.rewards["epsilan"][~0]

    print(f"\n| Mean Reward: {mean_reward} | Elite Mean Reward: {elite_mean_reward} |")
    print(f"| Loss: {round(loss, 3)} | Quantile: {q_param} | Epsilan: {round(epsilan, 3)} |\n")

    if len(self.rewards["q_param"]) > 0:
      _iter_rewards = {key: value[~0] for key, value in self.rewards.items()}
      _agent_stats = {key: value[~0] for key, value in self.agent.stats.items()}
      wandb.log({**_iter_rewards, **_agent_stats, **{"min_elite": min_elite, "max_elite": max_elite}})
      self.rewards = self.init_rewards()
      self.agent.stats = {}.fromkeys(self.agent.stats.keys(), [])

    if self.save_top_k:
      self.models.append({
          "state_dict": self.agent.state_dict(),
          "loss": round(loss, 3),
          "iter": self.current_iteration,
          "mean_reward": mean_reward
          })
      self.model_checkpoint()

  def train(self,
            discont_gamma: float = 1.0,
            q_param: Optional[float] = 0.65,
            iterations: Optional[int] = 100,
            trajectory_len: Optional[int] = 500,
            trajectory_n: Optional[int] = 32,
            save_top_k: int = None,
            save_by: Literal["loss", "mean_reward"] = "loss",
            mode: Literal["min", "max"] = "min"
            ):
    print(f"\n| Training initialized  🚀|\n{time.asctime()}\n")

    self.start_time = time.time()
    self.iterations = iterations
    self.save_top_k = save_top_k
    self.save_by = save_by
    self.discont_gamma = discont_gamma
    self.mode = mode

    for iteration in tqdm(range(iterations)):
      self.rewards["q_param"].append(q_param)
      self.current_iteration = iteration
      trajectories = self.get_trajectories(trajectory_len, trajectory_n)
      elite_trajectories = self.get_elite_trajectories(trajectories, q_param)

      if self.stage == "train":
        loss = self.agent.step(elite_trajectories)
        self.rewards["loss"].append(loss)
        self.rewards["epsilan"].append(self.agent.epsilan)
        self.on_epoch_end(trajectories, elite_trajectories)

    self.env.close()
    total_time = self.time()
    print(f"\n| Training finished  🚀|\nTotal time: {total_time} sec \ {round(total_time / 60, 3)} min\n")

# Стартовые функции

In [4]:
def seed_everything(seed: Optional[int] = 42) -> int:
    random.seed(seed)
    np.random.seed(seed)
    return seed

def setup_model(model: nn.Module, stage: str) -> nn.Module:
  stage_setup = True if stage == "train" else False
  for param in model.parameters():
    param.requires_grad = stage_setup
  model.train(stage_setup)
  return model

def check_sources():
    path = os.path.join(os.getcwd(), "models")
    if not os.path.exists(path):
        os.mkdir(path)
        
def on_startup():
  filterwarnings("ignore")  # Игнорировать предупреждения
  seed_everything(42)  # Фиксация датчика случайных чисел
  check_sources()

# Запуск обучения

In [5]:
# Создаём агента
agent = CrossEntropyAgent(
    state_dim=2,
    depth=[128, 128],
    action_n=1,
    dropout=None,
    batch_norm=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    epsilan=2,
    gamma=0.00005
)

# Создаём среду
env = SandBoxEnvironment(
    stage="train",
    agent=agent,  # Агент
    env_name="MountainCarContinuous-v0",  # Имя среды (v2)
)


if __name__ == "__main__":
  on_startup()

  wandb.init(
      project="deep-reinforcement-learning",
      name="practice-2.2 | train",
      tags=["iterations 200", "Adam", "lr 15e-4", "depth 1024x256x64", "trajectory_n 2_000", "trajectory_len 800", "Q 0.99", "gamma 0.00005"],
      job_type="train",
      reinit=True
      )

  # Запуск обучения
  env.train(
      discont_gamma=1.0,
      iterations=200,
      trajectory_n=2_000,
      trajectory_len=800,
      q_param=0.99,
      save_top_k=30,
      save_by="mean_reward",
      mode="max"
  )

wandb: Currently logged in as: entertomerci (cardio-sonix). Use `wandb login --relogin` to force relogin



| Training initialized  🚀|
Tue Oct 24 13:35:13 2023


  0%|          | 0/200 [00:00<?, ?it/s]


| Mean Reward: -49.23 | Elite Mean Reward: 63.31 |
| Loss: 0.65 | Quantile: 0.99 | Epsilan: 2.0 |


  0%|          | 1/200 [06:30<21:35:47, 390.69s/it]

...


| Mean Reward: -40.8 | Elite Mean Reward: 73.63 |
| Loss: 0.577 | Quantile: 0.99 | Epsilan: 1.694 |


 40%|████      | 81/200 [8:53:47<13:11:55, 399.29s/it]


| Mean Reward: -39.82 | Elite Mean Reward: 76.34 |
| Loss: 0.572 | Quantile: 0.99 | Epsilan: 1.687 |


 41%|████      | 82/200 [9:05:31<13:05:00, 399.16s/it]


KeyboardInterrupt: 

# Запуск тестирования

In [ ]:
# Создаём агента
agent = CrossEntropyAgent(
    state_dim=2,
    depth=[1024, 512, 64],
    action_n=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
)


state_dict =  torch.load("/home/merci/PycharmProjects/pytorchprojects/deep-RL-course/homework/practice2/models/loss=0.0_iter=61_mean_reward=-0.0.pth")
agent.load_state_dict(state_dict)


# Создаём среду
env = SandBoxEnvironment(
    stage="test",
    agent=agent,  # Агент
    env_name="MountainCarContinuous-v0",  # Имя среды (v2)
)


if __name__ == "__main__":
  on_startup()

  wandb.init(
      project="deep-reinforcement-learning",
      name="test runs",
      tags=["test", "rendering"],
      job_type="test"
      )

  # Запуск обучения
  env.train(
      iterations=1,  # Кол-во итераций 50
      trajectory_n=5,  # Кол-во траекторий 500
      trajectory_len=10_000,  # Длина траекторий 170
      save_top_k=None
  )